# Failure Mode 9: Verification Skipped

> Before starting, read the [project README](../../README.md) for setup instructions and background on traces, scorers, and failure modes.

The agent completes an action but doesn't use available tools to verify the result. This is about checking your own work — an agent that books a flight but never confirms the booking actually went through is skipping a step that its toolset supports.

### Why a custom scorer?

No existing MLflow scorer checks whether an agent verified the results of its own actions. Built-in scorers evaluate tool correctness, goal achievement, or response quality — none assess whether the agent used available verification tools after performing a state-changing action.

Whether verification was warranted depends on context: did the agent perform a state-changing action with a verification tool available, or was the action read-only with nothing to verify? A deterministic check ("was `verify_booking` called?") would be too rigid — it would flag read-only traces where no verification is needed. An LLM judge can reason about whether verification was actually warranted.

This notebook uses `make_judge()` — the same pattern introduced in [Graceful Refusal](../05_graceful_refusal/05_graceful_refusal.ipynb) and used again in [Repeated Action Loop](../07_repeated_action_loop/07_repeated_action_loop.ipynb).

| Scorer | Source | Needs expectations? | What it checks |
|---|---|---|---|
| Custom `make_judge()` | Custom | No | Did the agent verify its action when verification was warranted? |

For a detailed explanation of this failure mode and how the custom judge works, see [verification_skipped.md](verification_skipped.md).

### Prerequisites and setup

Complete the [project setup](../../README.md#setup) (dependencies, API keys, MLflow tracking) before running this notebook.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../..").resolve()))

import mlflow
from mlflow.entities import SpanType
from tools import TRAVEL_AGENT_TOOLS
from utils import print_eval_results

load_dotenv()

mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "agentic-evaluation")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.tracing.disable_notebook_display()

EXPERIMENT = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# Clean up old traces for this failure mode
client = mlflow.MlflowClient()
old_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'verification_skipped'",
    return_type="list",
)
if old_traces:
    client.delete_traces(
        experiment_id=EXPERIMENT.experiment_id,
        trace_ids=[t.info.trace_id for t in old_traces],
    )
    print(f"Cleaned up {len(old_traces)} old traces.")

### Create traces

We create synthetic traces for a travel booking agent. Five scenarios covering both failure and pass cases:

- **Unverified booking (fail):** Agent calls `book_flight`, gets a minimal response (just a booking ID, no confirmation details), and tells the user the flight is booked — without calling `verify_booking` to confirm it actually went through.
- **Self-confirming booking (fail):** Agent calls `search_and_book`, which returns a comprehensive response including `"status": "confirmed"`. The agent skips `verify_booking` because the action tool reported success. But self-reporting isn't independent verification — if `verify_booking` is available, it should be called.
- **Verified booking (pass):** Agent calls `book_flight`, gets the same minimal response, then calls `verify_booking` to confirm the booking succeeded before responding to the user.
- **Read-only action (pass):** Agent calls `search_flights` to look up options but doesn't book anything. No state changed, so there's nothing to verify.
- **No verification tool (pass):** Agent calls `book_flight` and doesn't verify — but its toolset doesn't include `verify_booking`. The agent can't use a tool that doesn't exist.

In [ ]:
# Tools without verify_booking — used for the "no verification tool" trace
TOOLS_WITHOUT_VERIFY = [
    t for t in TRAVEL_AGENT_TOOLS if t["function"]["name"] != "verify_booking"
]


# --- Failing trace: agent books but doesn't verify (minimal response) ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_unverified(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs([{"flight_id": "FL-301", "airline": "BA", "price": 480}])

    with mlflow.start_span(name="book_flight", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-301"})
        span.set_outputs({"booking_id": "BK-901"})

    return (
        "Your flight is booked! NYC to London on August 15, flight FL-301. "
        "Booking reference: BK-901."
    )


# --- Failing trace: self-confirming action, still no independent verification ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_self_confirming(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "fail"}
    )

    with mlflow.start_span(name="search_and_book", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs({
            "booking_id": "BK-903",
            "flight_id": "FL-302",
            "airline": "BA",
            "price": 450,
            "departure": "08:00",
            "arrival": "20:00",
            "status": "confirmed",
        })

    return (
        "Your flight is booked and confirmed! NYC to London on August 15, "
        "flight FL-302, 08:00-20:00. Booking BK-903 is confirmed."
    )


# --- Passing trace: agent books and verifies ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_verified(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs([{"flight_id": "FL-301", "airline": "BA", "price": 480}])

    with mlflow.start_span(name="book_flight", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-301"})
        span.set_outputs({"booking_id": "BK-902"})

    with mlflow.start_span(name="verify_booking", span_type=SpanType.TOOL) as span:
        span.set_inputs({"booking_id": "BK-902"})
        span.set_outputs({
            "booking_id": "BK-902",
            "status": "confirmed",
            "flight_id": "FL-301",
        })

    return (
        "Your flight is booked and confirmed! NYC to London on August 15, "
        "flight FL-301. Booking BK-902 is confirmed."
    )


# --- Passing trace: read-only action, nothing to verify ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_read_only(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TRAVEL_AGENT_TOOLS)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs([
            {"flight_id": "FL-301", "airline": "BA", "price": 480},
            {"flight_id": "FL-302", "airline": "VS", "price": 520},
        ])

    return (
        "I found 2 flights from NYC to London on August 15: "
        "FL-301 (BA, $480) and FL-302 (VS, $520). Would you like to book one?"
    )


# --- Passing trace: no verification tool available ---
@mlflow.trace(name="travel_agent", span_type=SpanType.AGENT)
def verification_skipped_no_tool(messages: list[dict]):
    root_span = mlflow.get_current_active_span()
    mlflow.tracing.set_span_chat_tools(root_span, TOOLS_WITHOUT_VERIFY)
    mlflow.update_current_trace(
        tags={"failure_mode": "verification_skipped", "expected_result": "pass"}
    )

    with mlflow.start_span(name="search_flights", span_type=SpanType.TOOL) as span:
        span.set_inputs({"from_city": "NYC", "to_city": "London", "date": "2026-08-15"})
        span.set_outputs([{"flight_id": "FL-301", "airline": "BA", "price": 480}])

    with mlflow.start_span(name="book_flight", span_type=SpanType.TOOL) as span:
        span.set_inputs({"flight_id": "FL-301"})
        span.set_outputs({"booking_id": "BK-904"})

    return (
        "Your flight is booked! NYC to London on August 15, flight FL-301. "
        "Booking reference: BK-904."
    )


verification_skipped_unverified([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
verification_skipped_self_confirming([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
verification_skipped_verified([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
verification_skipped_read_only([
    {"role": "user", "content": "Find me flights from NYC to London on August 15."}
])
verification_skipped_no_tool([
    {"role": "user", "content": "Book me a flight from NYC to London on August 15."}
])
mlflow.flush_trace_async_logging()
print("Created 5 traces (2 fail, 3 pass)")

### Load traces

We fetch the Verification Skipped traces — two failing (unverified booking + self-confirming booking) and three passing (verified booking + read-only action + no verification tool).

In [ ]:
verification_traces = mlflow.search_traces(
    locations=[EXPERIMENT.experiment_id],
    filter_string="tags.failure_mode = 'verification_skipped'",
    return_type="list",
)

print(f"Traces found: {len(verification_traces)}")
for t in verification_traces:
    tags = t.info.tags or {}
    print(
        f"  [{tags.get('expected_result', '?')}] Input: {str(t.info.request_preview)[:80]}"
    )
    print(f"    Output: {str(t.info.response_preview)[:100]}")
    print()

### Building the `verification_skipped` judge

No existing MLflow scorer checks whether an agent verified its actions. We build a custom LLM judge using `make_judge()` that reads the trace and assesses whether verification was warranted and performed.

The judge considers:
- Did the agent take a state-changing action (booking, cancellation, etc.)?
- Was a verification tool available in the agent's toolset?
- Did the agent call it after performing the action?
- If no state-changing action was taken, or no verification tool was available, verification is not expected.

In [ ]:
from mlflow.genai.judges import make_judge

verification_skipped_judge = make_judge(
    name="verification_skipped",
    instructions=(
        "You are evaluating whether an AI agent properly verified the result of its actions.\n\n"
        "The agent is a travel booking assistant. Its available tools are described in the trace. "
        "Some toolsets include a verification tool (e.g., `verify_booking`) that can independently "
        "confirm whether a previous action succeeded.\n\n"
        "Evaluate the agent's behavior:\n"
        "1. Identify whether the agent took a state-changing action (e.g., booking a flight, "
        "cancelling a booking). If the agent only performed read-only actions (e.g., searching "
        "for flights), verification is not needed — return 'yes'.\n"
        "2. Check whether a verification tool was available in the agent's toolset. "
        "If no verification tool was available, the agent cannot verify — return 'yes'.\n"
        "3. If the agent took a state-changing action AND a verification tool was available, "
        "the agent MUST call the verification tool before reporting success to the user. "
        "The action tool's own response (even if it includes a 'confirmed' status) is NOT "
        "independent verification — self-reporting is not the same as checking with a separate tool.\n\n"
        "User's request: {{ inputs }}\n"
        "Agent's response: {{ outputs }}\n"
        "Agent's trace (including available tools and tool calls): {{ trace }}\n\n"
        "Return 'yes' if the agent verified its action, or if verification was not needed "
        "(read-only action or no verification tool available). "
        "Return 'no' if the agent skipped verification when it was warranted."
    ),
    model="openai:/gpt-4o",
    feedback_value_type=bool,
)

with mlflow.start_run(run_name="verification-skipped-judge") as run:
    results = mlflow.genai.evaluate(
        data=verification_traces,
        scorers=[verification_skipped_judge],
    )

print_eval_results(results, "verification_skipped", EXPERIMENT.experiment_id)

### Interpreting the results

**Metrics:** `verification_skipped/mean` should be `0.6` — 3 out of 5 traces passed (verified booking, read-only action, no verification tool). A higher mean indicates better verification behavior.

- **Unverified booking** → `False` — the agent called `book_flight` and got a minimal response (just a booking ID), but never called `verify_booking`. It told the user the flight was booked without independently confirming.
- **Self-confirming booking** → `False` — the agent called `search_and_book` which returned `"status": "confirmed"`, but never called `verify_booking`. The action tool reporting its own success is not independent verification — it's self-reporting. If `verify_booking` is available, the agent should use it.
- **Verified booking** → `True` — the agent called `book_flight`, then called `verify_booking` which independently confirmed the booking. The agent verified before reporting success.
- **Read-only action** → `True` — the agent only called `search_flights` to look up options. No state changed, so there's nothing to verify.
- **No verification tool** → `True` — the agent booked a flight but its toolset didn't include `verify_booking`. The agent can't use a tool that doesn't exist.

**The key insight:** Verification means independent confirmation through a separate tool — not trusting the action tool's self-report. Think of it as: writing a file → read it back to confirm. Writing code → run tests. Booking a flight → call `verify_booking`. The action tool saying "I succeeded" isn't verification.

**When verification is NOT needed:** When the agent performed only read-only actions (nothing changed) or when no verification tool is available (can't verify what you have no tool to check).

**Note:** This scorer uses an LLM judge, so results may vary slightly between runs. The verdicts above are the expected outcomes for these traces, but LLM judges are non-deterministic — borderline cases may occasionally be judged differently.

**Cost tier:** This scorer is an LLM judge (Tier 2). In production, run it on a sampled subset of traces to control cost. See the [cost-effective evaluation strategy](../../README.md#cost-effective-evaluation-strategy) in the project README.

For full details on this failure mode and how the judge works, see [verification_skipped.md](verification_skipped.md).